In [1]:
from pathlib import Path
import scanpy as sc
import numpy as np
from scipy.stats import median_abs_deviation
import anndata as ad

In [2]:
def is_outlier(adata, metric: str, nmads: int):
    M = adata.obs[metric]
    outlier = (M < np.median(M) - nmads * median_abs_deviation(M)) | (
        np.median(M) + nmads * median_abs_deviation(M) < M
    )
    return outlier

In [3]:
def find_matrix_dir(sample_dir: Path) -> Path:
    candidates = [
        sample_dir / "outs/Gene/filtered",                 # STARsolo
        sample_dir / "outs/filtered_feature_bc_matrix",    # Cell Ranger (filtered)
    ]
    for p in candidates:
        if p.is_dir():
            return p
    raise FileNotFoundError(f"No 10x matrix dir under {sample_dir}")

In [4]:
def read_one_sample(sample_id: str) -> ad.AnnData:
    sdir = BASE / sample_id
    mdir = find_matrix_dir(sdir)

    adata = sc.read_10x_mtx(mdir, var_names='gene_symbols')
    adata.var_names_make_unique()

    adata.obs["barcode"] = adata.obs_names.astype(str)
    adata.obs["sample"]  = sample_id
    adata.obs_names = [f"{sample_id}_{bc}" for bc in adata.obs["barcode"]]

    g = adata.var_names.str.upper()
    adata.var["mt"]   = g.str.startswith("MT-")
    adata.var["ribo"] = g.str.startswith(("RPS", "RPL"))
    adata.var["hb"]   = g.str.match(r"^HB(?!P)")

    sc.pp.calculate_qc_metrics(
        adata, qc_vars=["mt", "ribo", "hb"], percent_top=None, inplace=True
    )
    return adata

In [5]:
BASE = Path("/home/sduan/scrup/alignments")

In [6]:
sample_ids = sorted([p.name for p in BASE.iterdir() if p.is_dir() and p.name.startswith("SRX")])

In [ ]:
sample_ids = sample_ids[0:200]
sample_ids = sample_ids[0:200]
sample_ids = sample_ids[0:200]
sample_ids = sample_ids[0:200]

In [ ]:
# adata = read_one_sample("SRX10152701")
sample_ids

In [ ]:
adatas = [read_one_sample(sid) for sid in sample_ids]
adata_all = ad.concat(adatas, join="outer", label="sample", keys=sample_ids, index_unique=None)

In [22]:
adata_all

AnnData object with n_obs × n_vars = 809428 × 54872
    obs: 'barcode', 'sample', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb'

In [7]:
import scrublet as scr

In [8]:
adata = sc.read("../test.h5ad")

In [9]:
adata

AnnData object with n_obs × n_vars = 1995 × 24245
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'scDblFinder.class', 'scDblFinder.score', 'ident'
    uns: 'X_name'

In [26]:
adata.X.sum(axis=0)

matrix([[ 0., 14.,  0., ...,  0.,  0.,  0.]], dtype=float32)

In [22]:
adata = read_one_sample("SRX10152701")